In [1]:
arquivo = r"C:\TCC\Documentos\Instancias\Instância Teste 50 jobs.txt"

In [2]:
from pyscipopt import Model, quicksum
from itertools import combinations
import time

In [3]:
def ler_instancia(nome_arquivo):
    with open(nome_arquivo, "r", encoding="utf-8") as f:
        linhas = [linha.strip() for linha in f if linha.strip()]

    peso = {}
    familia_produto = {}
    capacidade = {}
    tempo_proc = {}
    maquinas_familia = {}
    familia = {}
    produtos_por_familia = {}

    inst = None
    secao = None
    produto_id = 1

    for linha in linhas:

        if "NÚMERO DE PRODUTOS" in linha:
            texto, inst = linha.split(":")

        # identificar seção
        if "NOME DO PRODUTO" in linha:
            secao = "produtos"
            continue

        elif "MÁQUINAS/CAPACIDADE" in linha:
            secao = "maquinas"
            continue

        elif "FAMÍLIA/TEMPO" in linha:
            secao = "familias"
            continue

        elif "---" in linha:
            continue

        # -------------------
        # PRODUTOS
        # -------------------
        if secao == "produtos":
            nome, p, f = linha.split("/")

            peso[produto_id] = float(p)
            familia_produto[produto_id] = int(f)

            produto_id += 1

        # -------------------
        # MÁQUINAS
        # -------------------
        elif secao == "maquinas":
            maq, cap = linha.split("/")

            capacidade[maq] = float(cap)

        # -------------------
        # FAMÍLIAS
        # -------------------
        elif secao == "familias":
            fam, tempo, maq = linha.split("/")

            fam = int(fam)
            familia[fam] = fam
            tempo_proc[fam] = int(tempo)
            maquinas_familia[fam] = maq.split(",")

    for produto, fam in familia_produto.items():

        if fam not in produtos_por_familia:
            produtos_por_familia[fam] = []

        produtos_por_familia[fam].append(produto)

    return (
        inst,
        peso,
        familia_produto,
        capacidade,
        tempo_proc,
        maquinas_familia,
        familia,
        produtos_por_familia
    )



In [4]:
inst, peso, familia_produto, capacidade, tempo_proc, maquinas_familia, familia, produtos_por_familia = ler_instancia(arquivo)

mod = Model("Scheduling")

In [5]:
T = []
P = []
lotes_validos = []
A = []
maquinas = list(capacidade.keys())
produtos = list(peso.keys())

In [6]:
maior_capacidade = max(capacidade.values())

for fam, produtos_fam in produtos_por_familia.items():
    for r in range(1, len(produtos_fam) + 1):
        for lote in combinations(produtos_fam, r):

            if sum(peso[p] for p in lote) <= maior_capacidade:
                lotes_validos.append(lote)

#Binariza :)
for lotes in lotes_validos:
    linha = [1 if n in lotes else 0 for n in produtos]
    A.append(linha)

#tempos
for lote in lotes_validos:
    tempo_lote = tempo_proc[familia_produto[lote[0]]]
    T.append(tempo_lote)

#pesos
for lote in lotes_validos:
    peso_lote = sum(peso[p] for p in lote)
    P.append(peso_lote)



In [7]:
#Variáveis------------------------------------------------------------------------------

#(Se lote n vai para a máquina j)
y = [[None for j in range(len(maquinas))] for n in range(len(lotes_validos))]
for n in range(len(y)):
    for j in range(len(maquinas)):
        y[n][j] = mod.addVar(f"Y{n},{j}", "binary")

Cmax = mod.addVar("Cmax", "continuous")

In [8]:

inicio = time.time()

#Restrições-----------------------------------------------------------------------------

for i in range(len(produtos)):
    con1 = mod.addCons(quicksum(quicksum(y[n][j] * A[n][i] for j in range(len(maquinas))) for n in range(len(lotes_validos))) == 1)

for n, lote in enumerate(lotes_validos):
    for j, maquina in enumerate(maquinas):
        if maquina not in maquinas_familia[familia_produto[lote[0]]]:
            con2 = mod.addCons(y[n][j] == 0)

for j in range(len(maquinas)):
    con3 = mod.addCons(quicksum(y[n][j] * T[n] for n in range(len(lotes_validos))) <= Cmax)

for n in range(len(lotes_validos)):
    for j in range(len(maquinas)):
        con4 = mod.addCons(P[n] * y[n][j] <= capacidade[maquinas[j]])

#Função Objetivo-------------------------------------------------------------------------

mod.setObjective(Cmax, "minimize")

#Otimização------------------------------------------------------------------------------

mod.writeProblem(filename = f"Resultados\\resultado_{inst}.lp")

mod.optimize()

fim = time.time()

wrote problem to file c:\TCC\Resultados\resultado_50.lp


In [9]:

with open(f"Resultados\\dados_lotes_{inst}_produtos.txt", "w") as arquivo:

    for i, lote in enumerate(lotes_validos):

        familia_lote = familia_produto[lote[0]]

        arquivo.write(f"Lote {i}\n")
        arquivo.write(f"Família: {familia_lote}\n")
        arquivo.write(f"Produtos: {lote}\n")
        arquivo.write(f"Peso: {P[i]}\n")
        arquivo.write(f"Tempo: {T[i]}\n")

        linha_A = " ".join(map(str, A[i]))

        arquivo.write(f"A: {linha_A}\n")

        arquivo.write("\n")

mod.writeProblem(filename = f"Resultados\\resultado_{inst}.lp")

with open(f"Resultados\\dados_Y_{inst}_produtos.txt", "w") as arquivo_Y:

    for n in range(len(lotes_validos)):

        for j in range(len(maquinas)):

            valor = mod.getVal(y[n][j])

            arquivo_Y.write(
                f"Y[{n},{j}] = {valor}\n"
            )

wrote problem to file c:\TCC\Resultados\resultado_50.lp


In [10]:
valor = mod.getVal(Cmax)
tempo_total = fim - inicio

print(f"Tempo de execução: {tempo_total:.4f} segundos")
print(mod.getStatus())
print(f"Cmax = {valor} minutos")
print("")

for j in maquinas:
    print("-------------------------------------------------------")
    print(f"Máquina {j}: ")

print("-------------------------------------------------------")


Tempo de execução: 13.6567 segundos
optimal
Cmax = 178.0 minutos

-------------------------------------------------------
Máquina A: 
-------------------------------------------------------
Máquina B: 
-------------------------------------------------------
Máquina C: 
-------------------------------------------------------
Máquina D: 
-------------------------------------------------------
Máquina E: 
-------------------------------------------------------
Máquina F: 
-------------------------------------------------------
Máquina G: 
-------------------------------------------------------
